# Stage 8: Construction of Causal Dataset

This notebook constructs the balanced causal panel dataset used for
Difference-in-Differences and subsequent causal analyses.

The panel contains:

- Pre-outage period: 14 days before outage creation
- During-outage period: outage creation to repair completion
- Post-repair period: 14 days after repair
- Treatment/control indicators
- Treatment × post interaction
- Direct crime counts (0–100m)
- Displacement crime counts (100–250m)
- Spatial and outage covariates

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Stage 8: Construction of Causal Dataset")

Stage 8: Construction of Causal Dataset


In [2]:
# ---------------------------------------------------------
# File paths
# ---------------------------------------------------------

PROCESSED_DIR = Path("../data/processed")

CONTROL_FILE = PROCESSED_DIR / "control_area_pairs.parquet"
CRIME_FILE = PROCESSED_DIR / "clean_crime.parquet"
OUTAGE_FILE = PROCESSED_DIR / "clean_streetlights.parquet"

OUTPUT_FILE = PROCESSED_DIR / "causal_panel.parquet"

print("Control pairs:", CONTROL_FILE)
print("Crime data:", CRIME_FILE)
print("Streetlight data:", OUTAGE_FILE)
print("Output:", OUTPUT_FILE)

Control pairs: ..\data\processed\control_area_pairs.parquet
Crime data: ..\data\processed\clean_crime.parquet
Streetlight data: ..\data\processed\clean_streetlights.parquet
Output: ..\data\processed\causal_panel.parquet


In [3]:
control_pairs = pd.read_parquet(CONTROL_FILE)

print("Control pairs shape:", control_pairs.shape)
print("\nColumns:")
print(control_pairs.columns.tolist())

control_pairs.head()

Control pairs shape: (46124, 26)

Columns:
['treatment_key', 'created_date_left', 'closed_date_left', 'h3_res9_left', 'h3_res10_left', 'geometry', 'index_right', 'control_key', 'created_date_right', 'closed_date_right', 'h3_res9_right', 'h3_res10_right', 'control_geometry', 'distance_m', 'temporal_overlap', 'created_date', 'closed_date', 'outage_duration_hours', 'h3_res9', 'h3_res10', 'treatment_crime_100m', 'treatment_crime_250m', 'control_crime_100m', 'control_crime_250m', 'crime_difference_100m', 'crime_difference_250m']


,treatment_key,created_date_left,closed_date_left,h3_res9_left,h3_res10_left,geometry,index_right,control_key,created_date_right,closed_date_right,...,closed_date,outage_duration_hours,h3_res9,h3_res10,treatment_crime_100m,treatment_crime_250m,control_crime_100m,control_crime_250m,crime_difference_100m,crime_difference_250m
0,45284934,2020-01-01 11:04:00,2020-01-04 00:05:00,892a1072dcfffff,8a2a1072dce7fff,b'\x01\x01\x00\x00\x00c\xc7\xff\xff\xaf\x1f.A\...,98125,48296388,2020-12-01 16:20:00,2020-12-03 01:23:00,...,2020-01-04 00:05:00,61.016667,892a1072dcfffff,8a2a1072dce7fff,0,0,0,0,0,0
1,45284935,2020-01-01 06:55:00,2020-01-03 10:54:00,892a1072137ffff,8a2a1072134ffff,b'\x01\x01\x00\x00\x00\x1d9\x00\x00\n\xf3-Ay\x...,35867,64135128,2025-02-20 15:25:00,2025-02-26 03:25:00,...,2020-01-03 10:54:00,51.983333,892a1072137ffff,8a2a1072134ffff,0,0,0,0,0,0
2,45285212,2020-01-01 18:19:00,2020-01-06 09:16:00,892a10625cfffff,8a2a10625ccffff,"b'\x01\x01\x00\x00\x00\x96J\x02\x00\x96""-A\xdc...",32111,64845276,2025-05-04 21:41:00,2025-05-13 15:22:00,...,2020-01-06 09:16:00,110.950000,892a10625cfffff,8a2a10625ccffff,0,0,0,0,0,0
3,45285700,2020-01-01 20:11:00,2020-01-06 09:27:00,892a107560fffff,8a2a107560f7fff,b'\x01\x01\x00\x00\x00\xc3\xa0\x01\x00he-Az\xe...,97766,48430637,2020-12-16 09:54:00,2020-12-18 08:59:00,...,2020-01-06 09:27:00,109.266667,892a107560fffff,8a2a107560f7fff,0,0,0,0,0,0
4,45286571,2020-01-01 15:26:00,2020-01-03 12:15:00,892a1072d2bffff,8a2a1072d28ffff,"b'\x01\x01\x00\x00\x004\xbf\xff\xff\xf5"".A\xdc...",81636,53726952,2022-03-23 17:18:00,2022-03-24 11:27:00,...,2020-01-03 12:15:00,44.816667,892a1072d2bffff,8a2a1072d28ffff,0,0,0,0,0,0


In [4]:
crime = pd.read_parquet(CRIME_FILE)
outages = pd.read_parquet(OUTAGE_FILE)

print("Crime shape:", crime.shape)
print("Outage shape:", outages.shape)

Crime shape: (63195, 38)
Outage shape: (108030, 46)


In [5]:
print("Crime columns:")
print(crime.columns.tolist())

print("\nOutage columns:")
print(outages.columns.tolist())

Crime columns:
['cmplnt_num', 'cmplnt_fr_dt', 'cmplnt_fr_tm', 'cmplnt_to_dt', 'cmplnt_to_tm', 'addr_pct_cd', 'rpt_dt', 'ky_cd', 'ofns_desc', 'pd_cd', 'pd_desc', 'crm_atpt_cptd_cd', 'law_cat_cd', 'boro_nm', 'loc_of_occur_desc', 'prem_typ_desc', 'juris_desc', 'jurisdiction_code', 'parks_nm', 'hadevelopt', 'housing_psa', 'x_coord_cd', 'y_coord_cd', 'susp_age_group', 'susp_race', 'susp_sex', 'transit_district', 'latitude', 'longitude', 'lat_lon', 'patrol_boro', 'station_name', 'vic_age_group', 'vic_race', 'vic_sex', 'crime_datetime', 'is_nighttime', 'geometry']

Outage columns:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'facility_type', 'status', 'due_date', 'resolution_description', 'resolution_action_updated_date', 'community_

In [6]:
print("Control pair sample:")
display(
    control_pairs[
        [
            "treatment_key",
            "control_key",
            "created_date",
            "closed_date",
            "distance_m",
            "temporal_overlap",
            "treatment_crime_100m",
            "treatment_crime_250m",
            "control_crime_100m",
            "control_crime_250m"
        ]
    ].head()
)

Control pair sample:


,treatment_key,control_key,created_date,closed_date,distance_m,temporal_overlap,treatment_crime_100m,treatment_crime_250m,control_crime_100m,control_crime_250m
0,45284934,48296388,2020-01-01 11:04:00,2020-01-04 00:05:00,176.139150,False,0,0,0,0
1,45284935,64135128,2020-01-01 06:55:00,2020-01-03 10:54:00,132.457540,False,0,0,0,0
2,45285212,64845276,2020-01-01 18:19:00,2020-01-06 09:16:00,135.203550,False,0,0,0,0
3,45285700,48430637,2020-01-01 20:11:00,2020-01-06 09:27:00,106.470653,False,0,0,0,0
4,45286571,53726952,2020-01-01 15:26:00,2020-01-03 12:15:00,149.083869,False,0,0,0,0


In [7]:
print("Crime sample:")
display(crime.head())

Crime sample:


,cmplnt_num,cmplnt_fr_dt,cmplnt_fr_tm,cmplnt_to_dt,cmplnt_to_tm,addr_pct_cd,rpt_dt,ky_cd,ofns_desc,pd_cd,...,longitude,lat_lon,patrol_boro,station_name,vic_age_group,vic_race,vic_sex,crime_datetime,is_nighttime,geometry
0,318185372,2025-12-31T00:00:00.000,03:53:00,2025-12-31T00:00:00.000,04:25:00,106,2025-12-31T00:00:00.000,341,PETIT LARCENY,321.0,...,-73.824175,"(40.68623422, -73.82417476)",PATROL BORO QUEENS SOUTH,(null),UNKNOWN,UNKNOWN,D,2025-12-31 03:53:00,True,b'\x01\x01\x00\x00\x00\xe0\x18~G\xbftR\xc0\xe6...
1,318194633,2025-12-30T00:00:00.000,20:00:00,2025-12-31T00:00:00.000,14:00:00,67,2025-12-31T00:00:00.000,109,GRAND LARCENY,421.0,...,-73.944779,"(40.642343, -73.944779)",PATROL BORO BKLYN SOUTH,(null),45-64,BLACK,F,2025-12-30 20:00:00,True,b'\x01\x01\x00\x00\x00\xa5\xbcVBw|R\xc0B\xe8\x...
2,318205065,2025-12-31T00:00:00.000,18:00:00,2025-12-31T00:00:00.000,18:10:00,121,2025-12-31T00:00:00.000,109,GRAND LARCENY,439.0,...,-74.169068,"(40.582314, -74.169068)",PATROL BORO STATEN ISLAND,(null),25-44,ASIAN / PACIFIC ISLANDER,F,2025-12-31 18:00:00,True,b'\x01\x01\x00\x00\x005\xb3\x96\x02\xd2\x8aR\x...
3,318198685,2025-12-13T00:00:00.000,02:40:00,2025-12-13T00:00:00.000,03:11:00,81,2025-12-31T00:00:00.000,109,GRAND LARCENY,406.0,...,-73.929702,"(40.69386225, -73.92970228)",PATROL BORO BKLYN NORTH,KOSCIUSKO STREET,65+,BLACK HISPANIC,M,2025-12-13 02:40:00,True,b'\x01\x01\x00\x00\x00w\xe7\xfd=\x80{R\xc0\xe9...
4,318185349,2025-12-30T00:00:00.000,22:30:00,2025-12-31T00:00:00.000,08:00:00,71,2025-12-31T00:00:00.000,341,PETIT LARCENY,338.0,...,-73.948802,"(40.657196, -73.948802)",PATROL BORO BKLYN SOUTH,(null),65+,BLACK,M,2025-12-30 22:30:00,True,"b'\x01\x01\x00\x00\x00H\x18\x06,\xb9|R\xc0\xf0..."


In [9]:
print("Outage sample:")

print("\nAvailable outage columns:")
print(outages.columns.tolist())

display(
    outages[
        [
            "unique_key",
            "created_date",
            "closed_date",
            "outage_duration_hours",
            "latitude",
            "longitude",
            "borough"
        ]
    ].head()
)

Outage sample:

Available outage columns:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'facility_type', 'status', 'due_date', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'vehicle_type', 'taxi_company_borough', 'taxi_pick_up_location', 'bridge_highway_name', 'bridge_highway_direction', 'road_ramp', 'bridge_highway_segment', 'latitude', 'longitude', 'location', 'outage_duration_hours', 'geometry']


,unique_key,created_date,closed_date,outage_duration_hours,latitude,longitude,borough
0,70443136,2026-09-16 21:39:00,2026-09-17 09:34:00,11.916667,40.695643,-73.997382,BROOKLYN
1,70427817,2026-09-15 20:07:00,2026-09-17 09:17:00,37.166667,40.693572,-73.922746,BROOKLYN
2,70423298,2026-09-15 15:54:00,2026-09-17 09:18:00,41.400000,40.692119,-73.906600,BROOKLYN
3,70424836,2026-09-15 12:47:00,2026-09-18 09:15:00,68.466667,40.687139,-73.987430,BROOKLYN
4,70424835,2026-09-15 09:35:00,2026-09-17 09:32:00,47.950000,40.694984,-73.997699,BROOKLYN


In [10]:
print("Outage columns containing H3:")
print([
    col for col in outages.columns
    if "h3" in col.lower()
])

Outage columns containing H3:
[]


In [11]:
print("Crime columns:")
print(crime.columns.tolist())

print("\nCrime shape:")
print(crime.shape)

print("\nCrime sample:")
display(crime.head())

Crime columns:
['cmplnt_num', 'cmplnt_fr_dt', 'cmplnt_fr_tm', 'cmplnt_to_dt', 'cmplnt_to_tm', 'addr_pct_cd', 'rpt_dt', 'ky_cd', 'ofns_desc', 'pd_cd', 'pd_desc', 'crm_atpt_cptd_cd', 'law_cat_cd', 'boro_nm', 'loc_of_occur_desc', 'prem_typ_desc', 'juris_desc', 'jurisdiction_code', 'parks_nm', 'hadevelopt', 'housing_psa', 'x_coord_cd', 'y_coord_cd', 'susp_age_group', 'susp_race', 'susp_sex', 'transit_district', 'latitude', 'longitude', 'lat_lon', 'patrol_boro', 'station_name', 'vic_age_group', 'vic_race', 'vic_sex', 'crime_datetime', 'is_nighttime', 'geometry']

Crime shape:
(63195, 38)

Crime sample:


,cmplnt_num,cmplnt_fr_dt,cmplnt_fr_tm,cmplnt_to_dt,cmplnt_to_tm,addr_pct_cd,rpt_dt,ky_cd,ofns_desc,pd_cd,...,longitude,lat_lon,patrol_boro,station_name,vic_age_group,vic_race,vic_sex,crime_datetime,is_nighttime,geometry
0,318185372,2025-12-31T00:00:00.000,03:53:00,2025-12-31T00:00:00.000,04:25:00,106,2025-12-31T00:00:00.000,341,PETIT LARCENY,321.0,...,-73.824175,"(40.68623422, -73.82417476)",PATROL BORO QUEENS SOUTH,(null),UNKNOWN,UNKNOWN,D,2025-12-31 03:53:00,True,b'\x01\x01\x00\x00\x00\xe0\x18~G\xbftR\xc0\xe6...
1,318194633,2025-12-30T00:00:00.000,20:00:00,2025-12-31T00:00:00.000,14:00:00,67,2025-12-31T00:00:00.000,109,GRAND LARCENY,421.0,...,-73.944779,"(40.642343, -73.944779)",PATROL BORO BKLYN SOUTH,(null),45-64,BLACK,F,2025-12-30 20:00:00,True,b'\x01\x01\x00\x00\x00\xa5\xbcVBw|R\xc0B\xe8\x...
2,318205065,2025-12-31T00:00:00.000,18:00:00,2025-12-31T00:00:00.000,18:10:00,121,2025-12-31T00:00:00.000,109,GRAND LARCENY,439.0,...,-74.169068,"(40.582314, -74.169068)",PATROL BORO STATEN ISLAND,(null),25-44,ASIAN / PACIFIC ISLANDER,F,2025-12-31 18:00:00,True,b'\x01\x01\x00\x00\x005\xb3\x96\x02\xd2\x8aR\x...
3,318198685,2025-12-13T00:00:00.000,02:40:00,2025-12-13T00:00:00.000,03:11:00,81,2025-12-31T00:00:00.000,109,GRAND LARCENY,406.0,...,-73.929702,"(40.69386225, -73.92970228)",PATROL BORO BKLYN NORTH,KOSCIUSKO STREET,65+,BLACK HISPANIC,M,2025-12-13 02:40:00,True,b'\x01\x01\x00\x00\x00w\xe7\xfd=\x80{R\xc0\xe9...
4,318185349,2025-12-30T00:00:00.000,22:30:00,2025-12-31T00:00:00.000,08:00:00,71,2025-12-31T00:00:00.000,341,PETIT LARCENY,338.0,...,-73.948802,"(40.657196, -73.948802)",PATROL BORO BKLYN SOUTH,(null),65+,BLACK,M,2025-12-30 22:30:00,True,"b'\x01\x01\x00\x00\x00H\x18\x06,\xb9|R\xc0\xf0..."


In [12]:
print("\nCrime datetime-related columns:")
print([
    col for col in crime.columns
    if any(x in col.lower() for x in ["date", "time", "datetime"])
])

print("\nCrime spatial columns:")
print([
    col for col in crime.columns
    if any(x in col.lower() for x in ["lat", "lon", "geometry", "h3"])
])


Crime datetime-related columns:
['crime_datetime', 'is_nighttime']

Crime spatial columns:
['latitude', 'longitude', 'lat_lon', 'geometry']


In [14]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path

In [15]:
crime = pd.read_parquet(CRIME_FILE)

In [16]:
crime = gpd.read_parquet(CRIME_FILE)

print("Crime type:", type(crime))
print("Crime shape:", crime.shape)
print("Crime CRS:", crime.crs)

Crime type: <class 'geopandas.geodataframe.GeoDataFrame'>
Crime shape: (63195, 38)
Crime CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic lon

In [17]:
outages = pd.read_parquet(OUTAGE_FILE)

In [18]:
outages = gpd.read_parquet(OUTAGE_FILE)

print("Outage type:", type(outages))
print("Outage shape:", outages.shape)
print("Outage CRS:", outages.crs)

Outage type: <class 'geopandas.geodataframe.GeoDataFrame'>
Outage shape: (108030, 46)
Outage CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic

In [19]:
print("Crime records:", len(crime))
print("Missing crime_datetime:", crime["crime_datetime"].isna().sum())
print("Missing geometry:", crime.geometry.isna().sum())

print("\nCrime CRS:")
print(crime.crs)

print("\nGeometry types:")
print(crime.geometry.geom_type.value_counts())

Crime records: 63195
Missing crime_datetime: 0
Missing geometry: 0

Crime CRS:
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbre

In [20]:
CRS_PROJECTED = "EPSG:2263"

crime = crime.to_crs(CRS_PROJECTED)

print("Crime CRS after projection:")
print(crime.crs)

Crime CRS after projection:
EPSG:2263


In [21]:
outages["created_date"] = pd.to_datetime(
    outages["created_date"],
    errors="coerce"
)

outages["closed_date"] = pd.to_datetime(
    outages["closed_date"],
    errors="coerce"
)

outages = outages.dropna(
    subset=[
        "unique_key",
        "geometry",
        "created_date",
        "closed_date"
    ]
).copy()

outages = outages.to_crs(CRS_PROJECTED)

print("Prepared outage locations:", len(outages))
print("Outage CRS:", outages.crs)

Prepared outage locations: 108030
Outage CRS: EPSG:2263


In [22]:
print("========== SPATIAL VALIDATION ==========")

print("\nCrime:")
print("Type:", type(crime).__name__)
print("CRS:", crime.crs)
print("Records:", len(crime))
print("Missing geometry:", crime.geometry.isna().sum())

print("\nOutages:")
print("Type:", type(outages).__name__)
print("CRS:", outages.crs)
print("Records:", len(outages))
print("Missing geometry:", outages.geometry.isna().sum())

========== SPATIAL VALIDATION ==========

Crime:
Type: GeoDataFrame
CRS: EPSG:2263
Records: 63195
Missing geometry: 0

Outages:
Type: GeoDataFrame
CRS: EPSG:2263
Records: 108030
Missing geometry: 0


In [23]:
# ---------------------------------------------------------
# Attach treatment and control geometries
# ---------------------------------------------------------

outage_lookup = outages[
    [
        "unique_key",
        "geometry",
        "borough",
        "latitude",
        "longitude",
        "outage_duration_hours"
    ]
].copy()

outage_lookup["unique_key"] = outage_lookup["unique_key"].astype(int)

print("Outage lookup shape:", outage_lookup.shape)

Outage lookup shape: (108030, 6)


In [24]:
treatment_lookup = outage_lookup.rename(
    columns={
        "unique_key": "treatment_key",
        "geometry": "treatment_geometry",
        "borough": "treatment_borough",
        "latitude": "treatment_latitude",
        "longitude": "treatment_longitude",
        "outage_duration_hours": "treatment_duration_hours"
    }
)

control_lookup = outage_lookup.rename(
    columns={
        "unique_key": "control_key",
        "geometry": "control_geometry_actual",
        "borough": "control_borough",
        "latitude": "control_latitude",
        "longitude": "control_longitude",
        "outage_duration_hours": "control_duration_hours"
    }
)

In [25]:
pairs = control_pairs.merge(
    treatment_lookup,
    on="treatment_key",
    how="left"
)

pairs = pairs.merge(
    control_lookup,
    on="control_key",
    how="left"
)

print("Pairs shape:", pairs.shape)

print(
    "Missing treatment geometry:",
    pairs["treatment_geometry"].isna().sum()
)

print(
    "Missing control geometry:",
    pairs["control_geometry_actual"].isna().sum()
)

Pairs shape: (46124, 36)
Missing treatment geometry: 0
Missing control geometry: 0


In [26]:
def count_crimes_for_location(
    location_geometry,
    start_time,
    end_time,
    crime_gdf
):
    """
    Count nighttime crime events in:
      - direct ring: 0–100 m
      - displacement ring: 100–250 m

    for a specified temporal window.
    """

    # Candidate crimes within 250 metres using spatial index
    candidate_idx = crime_gdf.sindex.query(
        location_geometry.buffer(250),
        predicate="intersects"
    )

    candidates = crime_gdf.iloc[candidate_idx].copy()

    if candidates.empty:
        return 0, 0

    # Temporal filter
    candidates = candidates[
        (candidates["crime_datetime"] >= start_time)
        & (candidates["crime_datetime"] < end_time)
    ].copy()

    if candidates.empty:
        return 0, 0

    # Exact distance in metres
    distances = candidates.geometry.distance(location_geometry)

    direct_count = int((distances <= 100).sum())

    displacement_count = int(
        ((distances > 100) & (distances <= 250)).sum()
    )

    return direct_count, displacement_count

In [27]:
test_row = pairs.iloc[0]

print("Treatment:", test_row["treatment_key"])
print("Control:", test_row["control_key"])

print("Created:", test_row["created_date"])
print("Closed:", test_row["closed_date"])

Treatment: 45284934
Control: 48296388
Created: 2020-01-01 11:04:00
Closed: 2020-01-04 00:05:00


In [30]:
# ---------------------------------------------------------
# Recreate causal time windows directly on pairs
# ---------------------------------------------------------

pairs["created_date"] = pd.to_datetime(
    pairs["created_date"],
    errors="coerce"
)

pairs["closed_date"] = pd.to_datetime(
    pairs["closed_date"],
    errors="coerce"
)

# Remove pairs with invalid treatment dates
pairs = pairs.dropna(
    subset=["created_date", "closed_date"]
).copy()

# Pre-outage: 14 days before outage
pairs["pre_start"] = (
    pairs["created_date"] - pd.Timedelta(days=14)
)
pairs["pre_end"] = pairs["created_date"]

# During outage
pairs["during_start"] = pairs["created_date"]
pairs["during_end"] = pairs["closed_date"]

# Post-repair: 14 days after repair
pairs["post_start"] = pairs["closed_date"]
pairs["post_end"] = (
    pairs["closed_date"] + pd.Timedelta(days=14)
)

print("Causal windows created successfully.")
print("Pairs:", len(pairs))

display(
    pairs[
        [
            "treatment_key",
            "control_key",
            "created_date",
            "closed_date",
            "pre_start",
            "pre_end",
            "during_start",
            "during_end",
            "post_start",
            "post_end"
        ]
    ].head()
)

Causal windows created successfully.
Pairs: 46124


,treatment_key,control_key,created_date,closed_date,pre_start,pre_end,during_start,during_end,post_start,post_end
0,45284934,48296388,2020-01-01 11:04:00,2020-01-04 00:05:00,2019-12-18 11:04:00,2020-01-01 11:04:00,2020-01-01 11:04:00,2020-01-04 00:05:00,2020-01-04 00:05:00,2020-01-18 00:05:00
1,45284935,64135128,2020-01-01 06:55:00,2020-01-03 10:54:00,2019-12-18 06:55:00,2020-01-01 06:55:00,2020-01-01 06:55:00,2020-01-03 10:54:00,2020-01-03 10:54:00,2020-01-17 10:54:00
2,45285212,64845276,2020-01-01 18:19:00,2020-01-06 09:16:00,2019-12-18 18:19:00,2020-01-01 18:19:00,2020-01-01 18:19:00,2020-01-06 09:16:00,2020-01-06 09:16:00,2020-01-20 09:16:00
3,45285700,48430637,2020-01-01 20:11:00,2020-01-06 09:27:00,2019-12-18 20:11:00,2020-01-01 20:11:00,2020-01-01 20:11:00,2020-01-06 09:27:00,2020-01-06 09:27:00,2020-01-20 09:27:00
4,45286571,53726952,2020-01-01 15:26:00,2020-01-03 12:15:00,2019-12-18 15:26:00,2020-01-01 15:26:00,2020-01-01 15:26:00,2020-01-03 12:15:00,2020-01-03 12:15:00,2020-01-17 12:15:00


In [31]:
print("Window columns present:")

window_columns = [
    "pre_start",
    "pre_end",
    "during_start",
    "during_end",
    "post_start",
    "post_end"
]

print(
    pairs[window_columns].notna().all()
)

print("\nMissing window values:")
print(
    pairs[window_columns].isna().sum()
)

Window columns present:
pre_start       True
pre_end         True
during_start    True
during_end      True
post_start      True
post_end        True
dtype: bool

Missing window values:
pre_start       0
pre_end         0
during_start    0
during_end      0
post_start      0
post_end        0
dtype: int64


In [32]:
test_row = pairs.iloc[0]

print("Treatment:", test_row["treatment_key"])
print("Control:", test_row["control_key"])

print("\nPRE:")
print(test_row["pre_start"], "→", test_row["pre_end"])

print("\nDURING:")
print(test_row["during_start"], "→", test_row["during_end"])

print("\nPOST:")
print(test_row["post_start"], "→", test_row["post_end"])

Treatment: 45284934
Control: 48296388

PRE:
2019-12-18 11:04:00 → 2020-01-01 11:04:00

DURING:
2020-01-01 11:04:00 → 2020-01-04 00:05:00

POST:
2020-01-04 00:05:00 → 2020-01-18 00:05:00


In [33]:
pre_100, pre_250 = count_crimes_for_location(
    test_row["treatment_geometry"],
    test_row["pre_start"],
    test_row["pre_end"],
    crime
)

during_100, during_250 = count_crimes_for_location(
    test_row["treatment_geometry"],
    test_row["during_start"],
    test_row["during_end"],
    crime
)

post_100, post_250 = count_crimes_for_location(
    test_row["treatment_geometry"],
    test_row["post_start"],
    test_row["post_end"],
    crime
)

print("TREATMENT")
print("PRE:")
print("  0–100m:", pre_100)
print("  100–250m:", pre_250)

print("DURING:")
print("  0–100m:", during_100)
print("  100–250m:", during_250)

print("POST:")
print("  0–100m:", post_100)
print("  100–250m:", post_250)

TREATMENT
PRE:
  0–100m: 0
  100–250m: 0
DURING:
  0–100m: 0
  100–250m: 0
POST:
  0–100m: 0
  100–250m: 0


In [34]:
control_pre_100, control_pre_250 = count_crimes_for_location(
    test_row["control_geometry_actual"],
    test_row["pre_start"],
    test_row["pre_end"],
    crime
)

control_during_100, control_during_250 = count_crimes_for_location(
    test_row["control_geometry_actual"],
    test_row["during_start"],
    test_row["during_end"],
    crime
)

control_post_100, control_post_250 = count_crimes_for_location(
    test_row["control_geometry_actual"],
    test_row["post_start"],
    test_row["post_end"],
    crime
)

print("CONTROL")
print("PRE:")
print("  0–100m:", control_pre_100)
print("  100–250m:", control_pre_250)

print("DURING:")
print("  0–100m:", control_during_100)
print("  100–250m:", control_during_250)

print("POST:")
print("  0–100m:", control_post_100)
print("  100–250m:", control_post_250)

CONTROL
PRE:
  0–100m: 0
  100–250m: 0
DURING:
  0–100m: 0
  100–250m: 0
POST:
  0–100m: 0
  100–250m: 0


In [35]:
print("===== STAGE 7 vs STAGE 8 TEST =====")

print("\nTreatment 0–100m:")
print(
    "Stage 7:",
    test_row["treatment_crime_100m"]
)
print(
    "New:",
    during_100
)

print("\nTreatment 100–250m:")
print(
    "Stage 7:",
    test_row["treatment_crime_250m"]
)
print(
    "New:",
    during_250
)

print("\nControl 0–100m:")
print(
    "Stage 7:",
    test_row["control_crime_100m"]
)
print(
    "New:",
    control_during_100
)

print("\nControl 100–250m:")
print(
    "Stage 7:",
    test_row["control_crime_250m"]
)
print(
    "New:",
    control_during_250
)

===== STAGE 7 vs STAGE 8 TEST =====

Treatment 0–100m:
Stage 7: 0
New: 0

Treatment 100–250m:
Stage 7: 0
New: 0

Control 0–100m:
Stage 7: 0
New: 0

Control 100–250m:
Stage 7: 0
New: 0


In [36]:
# Find pairs with non-zero Stage 7 crime counts

nonzero_pairs = pairs[
    (pairs["treatment_crime_100m"] > 0) |
    (pairs["treatment_crime_250m"] > 0) |
    (pairs["control_crime_100m"] > 0) |
    (pairs["control_crime_250m"] > 0)
].copy()

print("Total pairs:", len(pairs))
print("Pairs with at least one crime:", len(nonzero_pairs))

display(
    nonzero_pairs[
        [
            "treatment_key",
            "control_key",
            "created_date",
            "closed_date",
            "treatment_crime_100m",
            "treatment_crime_250m",
            "control_crime_100m",
            "control_crime_250m"
        ]
    ].head(10)
)

Total pairs: 46124
Pairs with at least one crime: 1706


,treatment_key,control_key,created_date,closed_date,treatment_crime_100m,treatment_crime_250m,control_crime_100m,control_crime_250m
108,45332758,67059870,2020-01-07 21:01:00,2020-01-08 13:29:00,0,0,1,0
236,45380073,66648965,2020-01-13 16:47:00,2020-01-14 12:35:00,0,0,0,1
295,45397940,65841155,2020-01-15 04:43:00,2020-01-23 22:55:00,0,0,0,1
345,45418434,63734188,2020-01-18 21:59:00,2020-02-07 10:34:00,0,0,0,1
598,45523593,63734188,2020-02-01 18:25:00,2020-02-03 08:06:00,0,0,0,1
739,45588465,66738401,2020-02-10 02:16:00,2020-02-11 07:35:00,0,0,0,1
766,45599336,67157268,2020-02-12 17:28:00,2020-02-19 14:09:00,0,0,0,1
879,45653763,64485850,2020-02-19 12:49:00,2020-02-24 10:33:00,0,0,0,2
939,45693719,66632061,2020-02-25 20:29:00,2020-02-29 00:00:00,0,0,0,1
968,45707230,64485850,2020-02-27 09:06:00,2020-03-02 12:59:00,0,0,0,2


In [37]:
test_row = nonzero_pairs.iloc[0]

print("Treatment:", test_row["treatment_key"])
print("Control:", test_row["control_key"])

print("\nStage 7 counts:")
print(
    "Treatment 0–100m:",
    test_row["treatment_crime_100m"]
)
print(
    "Treatment 100–250m:",
    test_row["treatment_crime_250m"]
)
print(
    "Control 0–100m:",
    test_row["control_crime_100m"]
)
print(
    "Control 100–250m:",
    test_row["control_crime_250m"]
)

Treatment: 45332758
Control: 67059870

Stage 7 counts:
Treatment 0–100m: 0
Treatment 100–250m: 0
Control 0–100m: 1
Control 100–250m: 0


In [38]:
test_t_pre = count_crimes_for_location(
    test_row["treatment_geometry"],
    test_row["pre_start"],
    test_row["pre_end"],
    crime
)

test_t_during = count_crimes_for_location(
    test_row["treatment_geometry"],
    test_row["during_start"],
    test_row["during_end"],
    crime
)

test_t_post = count_crimes_for_location(
    test_row["treatment_geometry"],
    test_row["post_start"],
    test_row["post_end"],
    crime
)

print("\nTreatment:")
print("PRE:", test_t_pre)
print("DURING:", test_t_during)
print("POST:", test_t_post)


Treatment:
PRE: (0, 0)
DURING: (0, 0)
POST: (0, 0)


In [39]:
test_c_pre = count_crimes_for_location(
    test_row["control_geometry_actual"],
    test_row["pre_start"],
    test_row["pre_end"],
    crime
)

test_c_during = count_crimes_for_location(
    test_row["control_geometry_actual"],
    test_row["during_start"],
    test_row["during_end"],
    crime
)

test_c_post = count_crimes_for_location(
    test_row["control_geometry_actual"],
    test_row["post_start"],
    test_row["post_end"],
    crime
)

print("\nControl:")
print("PRE:", test_c_pre)
print("DURING:", test_c_during)
print("POST:", test_c_post)


Control:
PRE: (0, 0)
DURING: (0, 0)
POST: (0, 0)


In [40]:
# Inspect the first nonzero pair in detail

row = nonzero_pairs.iloc[0]

print("Treatment:", row["treatment_key"])
print("Control:", row["control_key"])
print("Created:", row["created_date"])
print("Closed:", row["closed_date"])

print("\nStage 7 counts:")
print("Treatment 100m:", row["treatment_crime_100m"])
print("Treatment 250m:", row["treatment_crime_250m"])
print("Control 100m:", row["control_crime_100m"])
print("Control 250m:", row["control_crime_250m"])

# Get the control geometry
control_geom = row["control_geometry_actual"]

# Find all crimes within 250m of the control
candidate_idx = crime.sindex.query(
    control_geom.buffer(250),
    predicate="intersects"
)

nearby_crimes = crime.iloc[candidate_idx].copy()

# Keep only crimes during the treatment outage
during_crimes = nearby_crimes[
    (nearby_crimes["crime_datetime"] >= row["created_date"]) &
    (nearby_crimes["crime_datetime"] < row["closed_date"])
].copy()

# Calculate exact distances
during_crimes["distance_m"] = during_crimes.geometry.distance(control_geom)

# Keep crimes within 250m
during_crimes = during_crimes[
    during_crimes["distance_m"] <= 250
].copy()

print("\nCrimes during outage within 250m of control:")
print("Count:", len(during_crimes))

display(
    during_crimes[
        [
            "crime_datetime",
            "latitude",
            "longitude",
            "distance_m"
        ]
    ].sort_values("distance_m")
)

Treatment: 45332758
Control: 67059870
Created: 2020-01-07 21:01:00
Closed: 2020-01-08 13:29:00

Stage 7 counts:
Treatment 100m: 0
Treatment 250m: 0
Control 100m: 1
Control 250m: 0

Crimes during outage within 250m of control:
Count: 0


,crime_datetime,latitude,longitude,distance_m


In [41]:
row = nonzero_pairs.iloc[0]

print("Treatment:", row["treatment_key"])
print("Control:", row["control_key"])

print("\nTreatment outage window:")
print("Start:", row["created_date"])
print("End:  ", row["closed_date"])

# -----------------------------------------
# Check treatment location during window
# -----------------------------------------

treatment_geom = row["treatment_geometry"]

candidate_idx = crime.sindex.query(
    treatment_geom.buffer(250),
    predicate="intersects"
)

treatment_crimes = crime.iloc[candidate_idx].copy()

treatment_crimes = treatment_crimes[
    (treatment_crimes["crime_datetime"] >= row["created_date"]) &
    (treatment_crimes["crime_datetime"] < row["closed_date"])
].copy()

treatment_crimes["distance_m"] = (
    treatment_crimes.geometry.distance(treatment_geom)
)

treatment_crimes = treatment_crimes[
    treatment_crimes["distance_m"] <= 250
].copy()

print("\nTreatment crimes DURING:")
print("Total within 250m:", len(treatment_crimes))
print("Within 100m:", (treatment_crimes["distance_m"] <= 100).sum())

display(
    treatment_crimes[
        ["crime_datetime", "latitude", "longitude", "distance_m"]
    ].sort_values("distance_m")
)


# -----------------------------------------
# Check control location during SAME window
# -----------------------------------------

control_geom = row["control_geometry_actual"]

candidate_idx = crime.sindex.query(
    control_geom.buffer(250),
    predicate="intersects"
)

control_crimes = crime.iloc[candidate_idx].copy()

control_crimes = control_crimes[
    (control_crimes["crime_datetime"] >= row["created_date"]) &
    (control_crimes["crime_datetime"] < row["closed_date"])
].copy()

control_crimes["distance_m"] = (
    control_crimes.geometry.distance(control_geom)
)

control_crimes = control_crimes[
    control_crimes["distance_m"] <= 250
].copy()

print("\nControl crimes DURING SAME treatment window:")
print("Total within 250m:", len(control_crimes))
print("Within 100m:", (control_crimes["distance_m"] <= 100).sum())

display(
    control_crimes[
        ["crime_datetime", "latitude", "longitude", "distance_m"]
    ].sort_values("distance_m")
)

Treatment: 45332758
Control: 67059870

Treatment outage window:
Start: 2020-01-07 21:01:00
End:   2020-01-08 13:29:00

Treatment crimes DURING:
Total within 250m: 0
Within 100m: 0


,crime_datetime,latitude,longitude,distance_m



Control crimes DURING SAME treatment window:
Total within 250m: 0
Within 100m: 0


,crime_datetime,latitude,longitude,distance_m


In [42]:
# Find a pair with a nonzero Stage 7 treatment crime count

treatment_nonzero = pairs[
    (pairs["treatment_crime_100m"] > 0) |
    (pairs["treatment_crime_250m"] > 0)
].copy()

print("Pairs with nonzero treatment crime:", len(treatment_nonzero))

display(
    treatment_nonzero[
        [
            "treatment_key",
            "control_key",
            "created_date",
            "closed_date",
            "treatment_crime_100m",
            "treatment_crime_250m",
            "control_crime_100m",
            "control_crime_250m",
        ]
    ].head(10)
)

Pairs with nonzero treatment crime: 912


,treatment_key,control_key,created_date,closed_date,treatment_crime_100m,treatment_crime_250m,control_crime_100m,control_crime_250m
8515,51328039,66493761,2021-07-24 21:08:00,2022-06-23 11:37:00,1,0,0,4
10497,52551934,59522070,2021-11-17 10:45:00,2022-05-05 12:38:00,0,1,0,0
12491,53509910,63102998,2022-03-02 10:29:00,2022-05-18 13:09:00,0,1,0,0
12492,53509925,63102998,2022-03-02 10:28:00,2023-01-09 10:10:00,0,1,0,0
13358,53974584,63102998,2022-04-21 12:58:00,2022-07-15 07:47:00,0,1,0,0
14946,54996866,53155784,2022-08-02 14:05:00,2022-09-08 10:16:00,0,1,0,0
17315,56253315,54097365,2022-12-14 20:08:00,2023-04-24 19:57:00,0,1,0,0
21465,59017083,69792597,2023-10-04 21:06:00,2024-06-07 13:31:00,0,1,0,0
21629,59092028,50540531,2023-10-12 20:57:00,2024-03-12 11:09:00,0,1,0,0
25315,60906107,51084019,2024-04-18 14:23:00,2024-08-29 18:04:00,0,1,0,0


In [43]:
# Validate the first nonzero treatment pair

row = treatment_nonzero.iloc[0]

print("Treatment:", row["treatment_key"])
print("Control:", row["control_key"])

print("\nStage 7 treatment counts:")
print("100m:", row["treatment_crime_100m"])
print("250m:", row["treatment_crime_250m"])

print("\nTreatment outage window:")
print("Start:", row["created_date"])
print("End:  ", row["closed_date"])

treatment_geom = row["treatment_geometry"]

candidate_idx = crime.sindex.query(
    treatment_geom.buffer(250),
    predicate="intersects"
)

test_crimes = crime.iloc[candidate_idx].copy()

test_crimes = test_crimes[
    (test_crimes["crime_datetime"] >= row["created_date"]) &
    (test_crimes["crime_datetime"] < row["closed_date"])
].copy()

test_crimes["distance_m"] = (
    test_crimes.geometry.distance(treatment_geom)
)

test_crimes = test_crimes[
    test_crimes["distance_m"] <= 250
].copy()

print("\nStage 8 recalculated treatment counts:")
print("100m:", (test_crimes["distance_m"] <= 100).sum())
print(
    "100-250m:",
    (
        (test_crimes["distance_m"] > 100) &
        (test_crimes["distance_m"] <= 250)
    ).sum()
)

display(
    test_crimes[
        ["crime_datetime", "latitude", "longitude", "distance_m"]
    ].sort_values("distance_m")
)

Treatment: 51328039
Control: 66493761

Stage 7 treatment counts:
100m: 1
250m: 0

Treatment outage window:
Start: 2021-07-24 21:08:00
End:   2022-06-23 11:37:00

Stage 8 recalculated treatment counts:
100m: 1
100-250m: 0


,crime_datetime,latitude,longitude,distance_m
49174,2021-12-01 00:01:00,40.815699,-73.924934,34.150561


In [ ]:
row = nonzero_pairs.iloc[0]

print("Treatment:", row["treatment_key"])
print("Control:", row["control_key"])

print("\nTreatment outage window:")
print("Start:", row["created_date"])
print("End:  ", row["closed_date"])

# -----------------------------------------
# Check treatment location during window
# -----------------------------------------

treatment_geom = row["treatment_geometry"]

candidate_idx = crime.sindex.query(
    treatment_geom.buffer(250),
    predicate="intersects"
)

treatment_crimes = crime.iloc[candidate_idx].copy()

treatment_crimes = treatment_crimes[
    (treatment_crimes["crime_datetime"] >= row["created_date"]) &
    (treatment_crimes["crime_datetime"] < row["closed_date"])
].copy()

treatment_crimes["distance_m"] = (
    treatment_crimes.geometry.distance(treatment_geom)
)

treatment_crimes = treatment_crimes[
    treatment_crimes["distance_m"] <= 250
].copy()

print("\nTreatment crimes DURING:")
print("Total within 250m:", len(treatment_crimes))
print("Within 100m:", (treatment_crimes["distance_m"] <= 100).sum())

display(
    treatment_crimes[
        ["crime_datetime", "latitude", "longitude", "distance_m"]
    ].sort_values("distance_m")
)


# -----------------------------------------
# Check control location during SAME window
# -----------------------------------------

control_geom = row["control_geometry_actual"]

candidate_idx = crime.sindex.query(
    control_geom.buffer(250),
    predicate="intersects"
)

control_crimes = crime.iloc[candidate_idx].copy()

control_crimes = control_crimes[
    (control_crimes["crime_datetime"] >= row["created_date"]) &
    (control_crimes["crime_datetime"] < row["closed_date"])
].copy()

control_crimes["distance_m"] = (
    control_crimes.geometry.distance(control_geom)
)

control_crimes = control_crimes[
    control_crimes["distance_m"] <= 250
].copy()

print("\nControl crimes DURING SAME treatment window:")
print("Total within 250m:", len(control_crimes))
print("Within 100m:", (control_crimes["distance_m"] <= 100).sum())

display(
    control_crimes[
        ["crime_datetime", "latitude", "longitude", "distance_m"]
    ].sort_values("distance_m")
)

Treatment: 45332758
Control: 67059870

Treatment outage window:
Start: 2020-01-07 21:01:00
End:   2020-01-08 13:29:00

Treatment crimes DURING:
Total within 250m: 0
Within 100m: 0


,crime_datetime,latitude,longitude,distance_m



Control crimes DURING SAME treatment window:
Total within 250m: 0
Within 100m: 0


,crime_datetime,latitude,longitude,distance_m


In [44]:
# Find pairs with a nonzero Stage 7 treatment crime count

treatment_nonzero = pairs[
    (pairs["treatment_crime_100m"] > 0) |
    (pairs["treatment_crime_250m"] > 0)
].copy()

print("Pairs with nonzero treatment crime:", len(treatment_nonzero))

display(
    treatment_nonzero[
        [
            "treatment_key",
            "control_key",
            "created_date",
            "closed_date",
            "treatment_crime_100m",
            "treatment_crime_250m",
            "control_crime_100m",
            "control_crime_250m",
        ]
    ].head(10)
)

Pairs with nonzero treatment crime: 912


,treatment_key,control_key,created_date,closed_date,treatment_crime_100m,treatment_crime_250m,control_crime_100m,control_crime_250m
8515,51328039,66493761,2021-07-24 21:08:00,2022-06-23 11:37:00,1,0,0,4
10497,52551934,59522070,2021-11-17 10:45:00,2022-05-05 12:38:00,0,1,0,0
12491,53509910,63102998,2022-03-02 10:29:00,2022-05-18 13:09:00,0,1,0,0
12492,53509925,63102998,2022-03-02 10:28:00,2023-01-09 10:10:00,0,1,0,0
13358,53974584,63102998,2022-04-21 12:58:00,2022-07-15 07:47:00,0,1,0,0
14946,54996866,53155784,2022-08-02 14:05:00,2022-09-08 10:16:00,0,1,0,0
17315,56253315,54097365,2022-12-14 20:08:00,2023-04-24 19:57:00,0,1,0,0
21465,59017083,69792597,2023-10-04 21:06:00,2024-06-07 13:31:00,0,1,0,0
21629,59092028,50540531,2023-10-12 20:57:00,2024-03-12 11:09:00,0,1,0,0
25315,60906107,51084019,2024-04-18 14:23:00,2024-08-29 18:04:00,0,1,0,0


In [45]:
# Validate the first pair with a nonzero treatment crime count

row = treatment_nonzero.iloc[0]

print("Treatment:", row["treatment_key"])
print("Control:", row["control_key"])

print("\nStage 7 treatment counts:")
print("100m:", row["treatment_crime_100m"])
print("250m:", row["treatment_crime_250m"])

print("\nTreatment outage window:")
print("Start:", row["created_date"])
print("End:  ", row["closed_date"])

treatment_geom = row["treatment_geometry"]

candidate_idx = crime.sindex.query(
    treatment_geom.buffer(250),
    predicate="intersects"
)

test_crimes = crime.iloc[candidate_idx].copy()

test_crimes = test_crimes[
    (test_crimes["crime_datetime"] >= row["created_date"]) &
    (test_crimes["crime_datetime"] < row["closed_date"])
].copy()

test_crimes["distance_m"] = (
    test_crimes.geometry.distance(treatment_geom)
)

test_crimes = test_crimes[
    test_crimes["distance_m"] <= 250
].copy()

print("\nStage 8 recalculated treatment counts:")
print("100m:", (test_crimes["distance_m"] <= 100).sum())

print(
    "100-250m:",
    (
        (test_crimes["distance_m"] > 100) &
        (test_crimes["distance_m"] <= 250)
    ).sum()
)

display(
    test_crimes[
        ["crime_datetime", "latitude", "longitude", "distance_m"]
    ].sort_values("distance_m")
)

Treatment: 51328039
Control: 66493761

Stage 7 treatment counts:
100m: 1
250m: 0

Treatment outage window:
Start: 2021-07-24 21:08:00
End:   2022-06-23 11:37:00

Stage 8 recalculated treatment counts:
100m: 1
100-250m: 0


,crime_datetime,latitude,longitude,distance_m
49174,2021-12-01 00:01:00,40.815699,-73.924934,34.150561


In [46]:
# ============================================================
# FULL STAGE 8 CRIME COUNT CALCULATION
# ============================================================

import numpy as np
from scipy.spatial import cKDTree
import time

print("Preparing crime spatial index...")

# ------------------------------------------------------------
# 1. Make sure crime dates are datetime
# ------------------------------------------------------------

crime["crime_datetime"] = pd.to_datetime(
    crime["crime_datetime"],
    errors="coerce"
)

crime = crime.dropna(
    subset=["crime_datetime", "geometry"]
).copy()


# ------------------------------------------------------------
# 2. Crime coordinates
#    EPSG:2263 = metres
# ------------------------------------------------------------

crime_x = crime.geometry.x.to_numpy()
crime_y = crime.geometry.y.to_numpy()

crime_coords = np.column_stack(
    [crime_x, crime_y]
)

crime_tree = cKDTree(crime_coords)

print("Crime records:", len(crime))
print("Crime spatial index ready.")


# ------------------------------------------------------------
# 3. Prepare outage coordinates
# ------------------------------------------------------------

outages = outages.to_crs("EPSG:2263").copy()

outage_coords = (
    outages
    .set_index("unique_key")
    .geometry
)

outage_xy = {
    int(key): (geom.x, geom.y)
    for key, geom in outage_coords.items()
    if geom is not None
}


# ------------------------------------------------------------
# 4. Get unique locations that we need
# ------------------------------------------------------------

treatment_keys = pairs["treatment_key"].astype(int).unique()
control_keys = pairs["control_key"].astype(int).unique()

location_keys = np.unique(
    np.concatenate([
        treatment_keys,
        control_keys
    ])
)

print("Unique locations required:", len(location_keys))


# ------------------------------------------------------------
# 5. Find crimes within 250m ONCE per location
# ------------------------------------------------------------

print("Finding nearby crimes for each unique location...")

start_time = time.time()

nearby_crimes = {}

for i, key in enumerate(location_keys):

    coord = outage_xy.get(int(key))

    if coord is None:
        nearby_crimes[int(key)] = np.array([], dtype=int)
        continue

    indices = crime_tree.query_ball_point(
        coord,
        r=250
    )

    nearby_crimes[int(key)] = np.asarray(
        indices,
        dtype=int
    )

    if (i + 1) % 5000 == 0:
        print(
            f"Processed {i + 1:,} / "
            f"{len(location_keys):,} locations"
        )

elapsed = time.time() - start_time

print(
    f"Spatial lookup completed in {elapsed:.1f} seconds"
)

Preparing crime spatial index...
Crime records: 63195
Crime spatial index ready.
Unique locations required: 46124
Finding nearby crimes for each unique location...
Processed 5,000 / 46,124 locations
Processed 10,000 / 46,124 locations
Processed 15,000 / 46,124 locations
Processed 20,000 / 46,124 locations
Processed 25,000 / 46,124 locations
Processed 30,000 / 46,124 locations
Processed 35,000 / 46,124 locations
Processed 40,000 / 46,124 locations
Processed 45,000 / 46,124 locations
Spatial lookup completed in 0.2 seconds


In [47]:
# ============================================================
# CALCULATE PRE / DURING / POST CRIME COUNTS
# ============================================================

crime_times = crime["crime_datetime"].to_numpy()

def count_from_indices(
    crime_indices,
    start_time,
    end_time
):
    """
    Count crimes in:
      0-100m
      100-250m

    during the specified time window.
    """

    if len(crime_indices) == 0:
        return 0, 0

    selected_times = crime_times[crime_indices]

    time_mask = (
        (selected_times >= np.datetime64(start_time)) &
        (selected_times < np.datetime64(end_time))
    )

    if not time_mask.any():
        return 0, 0

    selected_indices = crime_indices[time_mask]

    dx = (
        crime_x[selected_indices]
        - 0
    )

    dy = (
        crime_y[selected_indices]
        - 0
    )

    # We need the location coordinates separately,
    # so this function will be called with them below.
    return selected_indices


def count_crimes(
    location_key,
    location_coord,
    start_time,
    end_time
):
    """
    Count crimes within:
      0-100m
      100-250m

    for a specific location and time window.
    """

    crime_indices = nearby_crimes.get(
        int(location_key),
        np.array([], dtype=int)
    )

    if len(crime_indices) == 0:
        return 0, 0

    # Temporal filter
    selected_times = crime_times[crime_indices]

    time_mask = (
        (selected_times >= np.datetime64(start_time)) &
        (selected_times < np.datetime64(end_time))
    )

    if not time_mask.any():
        return 0, 0

    selected_indices = crime_indices[time_mask]

    # Spatial distance
    selected_x = crime_x[selected_indices]
    selected_y = crime_y[selected_indices]

    dx = selected_x - location_coord[0]
    dy = selected_y - location_coord[1]

    distances = np.sqrt(
        dx * dx + dy * dy
    )

    direct_count = int(
        (distances <= 100).sum()
    )

    displacement_count = int(
        (
            (distances > 100) &
            (distances <= 250)
        ).sum()
    )

    return direct_count, displacement_count


# ------------------------------------------------------------
# Calculate counts for every pair
# ------------------------------------------------------------

print("Calculating causal crime counts...")

start_time = time.time()

results = []

for i, row in pairs.iterrows():

    treatment_key = int(row["treatment_key"])
    control_key = int(row["control_key"])

    treatment_coord = outage_xy[treatment_key]
    control_coord = outage_xy[control_key]

    # --------------------------------------------------------
    # PRE
    # --------------------------------------------------------

    treatment_pre_100, treatment_pre_250 = count_crimes(
        treatment_key,
        treatment_coord,
        row["pre_start"],
        row["pre_end"]
    )

    control_pre_100, control_pre_250 = count_crimes(
        control_key,
        control_coord,
        row["pre_start"],
        row["pre_end"]
    )

    # --------------------------------------------------------
    # DURING
    # --------------------------------------------------------

    treatment_during_100, treatment_during_250 = count_crimes(
        treatment_key,
        treatment_coord,
        row["during_start"],
        row["during_end"]
    )

    control_during_100, control_during_250 = count_crimes(
        control_key,
        control_coord,
        row["during_start"],
        row["during_end"]
    )

    # --------------------------------------------------------
    # POST
    # --------------------------------------------------------

    treatment_post_100, treatment_post_250 = count_crimes(
        treatment_key,
        treatment_coord,
        row["post_start"],
        row["post_end"]
    )

    control_post_100, control_post_250 = count_crimes(
        control_key,
        control_coord,
        row["post_start"],
        row["post_end"]
    )

    results.append({
        "treatment_pre_100m": treatment_pre_100,
        "treatment_pre_250m": treatment_pre_250,

        "treatment_during_100m": treatment_during_100,
        "treatment_during_250m": treatment_during_250,

        "treatment_post_100m": treatment_post_100,
        "treatment_post_250m": treatment_post_250,

        "control_pre_100m": control_pre_100,
        "control_pre_250m": control_pre_250,

        "control_during_100m": control_during_100,
        "control_during_250m": control_during_250,

        "control_post_100m": control_post_100,
        "control_post_250m": control_post_250,
    })

    if len(results) % 5000 == 0:
        print(
            f"Processed {len(results):,} / "
            f"{len(pairs):,} pairs"
        )


# ------------------------------------------------------------
# Add results to pairs
# ------------------------------------------------------------

crime_counts = pd.DataFrame(
    results,
    index=pairs.index
)

pairs = pd.concat(
    [
        pairs,
        crime_counts
    ],
    axis=1
)

elapsed = time.time() - start_time

print(
    f"\nCrime-count calculation completed in "
    f"{elapsed:.1f} seconds"
)

print("Final rows:", len(pairs))

Calculating causal crime counts...
Processed 5,000 / 46,124 pairs
Processed 10,000 / 46,124 pairs
Processed 15,000 / 46,124 pairs
Processed 20,000 / 46,124 pairs
Processed 25,000 / 46,124 pairs
Processed 30,000 / 46,124 pairs
Processed 35,000 / 46,124 pairs
Processed 40,000 / 46,124 pairs
Processed 45,000 / 46,124 pairs

Crime-count calculation completed in 3.0 seconds
Final rows: 46124


In [48]:
# ============================================================
# VALIDATE STAGE 8 AGAINST STAGE 7
# ============================================================

# Stage 7 100m count vs Stage 8 DURING 100m count
pairs["treatment_100m_match"] = (
    pairs["treatment_crime_100m"]
    ==
    pairs["treatment_during_100m"]
)

# Stage 7 100-250m count vs Stage 8 DURING 100-250m count
#
# Stage 7's 250m value represents the displacement ring
# because the Stage 6 fields are already separated into
# 100m and 250m outcomes.
pairs["treatment_250m_match"] = (
    pairs["treatment_crime_250m"]
    ==
    pairs["treatment_during_250m"]
)

print(
    "Treatment 100m matches:",
    pairs["treatment_100m_match"].sum(),
    "/",
    len(pairs)
)

print(
    "Treatment 100-250m matches:",
    pairs["treatment_250m_match"].sum(),
    "/",
    len(pairs)
)

print(
    "100m mismatches:",
    (~pairs["treatment_100m_match"]).sum()
)

print(
    "100-250m mismatches:",
    (~pairs["treatment_250m_match"]).sum()
)

Treatment 100m matches: 46123 / 46124
Treatment 100-250m matches: 46123 / 46124
100m mismatches: 1
100-250m mismatches: 1


In [49]:
# Show the exact treatment-count mismatches

mismatches = pairs[
    (~pairs["treatment_100m_match"]) |
    (~pairs["treatment_250m_match"])
].copy()

print("Number of mismatching pairs:", len(mismatches))

display(
    mismatches[
        [
            "treatment_key",
            "control_key",
            "created_date",
            "closed_date",

            "treatment_crime_100m",
            "treatment_during_100m",

            "treatment_crime_250m",
            "treatment_during_250m",
        ]
    ]
)

Number of mismatching pairs: 1


,treatment_key,control_key,created_date,closed_date,treatment_crime_100m,treatment_during_100m,treatment_crime_250m,treatment_during_250m
37066,66314247,69221508,2025-09-29 14:17:00,2025-10-15 14:29:00,0,1,1,0


In [50]:
# Inspect the actual crimes for the mismatching treatment

row = mismatches.iloc[0]

print("Treatment:", row["treatment_key"])
print("Control:", row["control_key"])

print("\nStage 7 counts:")
print("100m:", row["treatment_crime_100m"])
print("250m:", row["treatment_crime_250m"])

print("\nStage 8 counts:")
print("100m:", row["treatment_during_100m"])
print("100-250m:", row["treatment_during_250m"])

print("\nWindow:")
print(row["created_date"], "to", row["closed_date"])

treatment_geom = row["treatment_geometry"]

candidate_idx = crime.sindex.query(
    treatment_geom.buffer(250),
    predicate="intersects"
)

check_crimes = crime.iloc[candidate_idx].copy()

check_crimes = check_crimes[
    (check_crimes["crime_datetime"] >= row["created_date"]) &
    (check_crimes["crime_datetime"] < row["closed_date"])
].copy()

check_crimes["distance_m"] = (
    check_crimes.geometry.distance(treatment_geom)
)

check_crimes = check_crimes[
    check_crimes["distance_m"] <= 250
].copy()

display(
    check_crimes[
        [
            "crime_datetime",
            "latitude",
            "longitude",
            "distance_m"
        ]
    ].sort_values("distance_m")
)

Treatment: 66314247
Control: 69221508

Stage 7 counts:
100m: 0
250m: 1

Stage 8 counts:
100m: 1
100-250m: 0

Window:
2025-09-29 14:17:00 to 2025-10-15 14:29:00


,crime_datetime,latitude,longitude,distance_m
16387,2025-10-07 19:00:00,40.875616,-73.870317,99.989463


In [51]:
# Correct Stage 7 vs Stage 8 validation

pairs["treatment_100m_match"] = (
    pairs["treatment_crime_100m"]
    ==
    pairs["treatment_during_100m"]
)

pairs["treatment_250m_cumulative_match"] = (
    pairs["treatment_crime_250m"]
    ==
    (
        pairs["treatment_during_100m"]
        +
        pairs["treatment_during_250m"]
    )
)

print(
    "Stage 7 100m matches:",
    pairs["treatment_100m_match"].sum(),
    "/",
    len(pairs)
)

print(
    "Stage 7 cumulative 250m matches:",
    pairs["treatment_250m_cumulative_match"].sum(),
    "/",
    len(pairs)
)

print(
    "100m mismatches:",
    (~pairs["treatment_100m_match"]).sum()
)

print(
    "250m cumulative mismatches:",
    (~pairs["treatment_250m_cumulative_match"]).sum()
)

Stage 7 100m matches: 46123 / 46124
Stage 7 cumulative 250m matches: 45927 / 46124
100m mismatches: 1
250m cumulative mismatches: 197


In [52]:
# ============================================================
# STAGE 8 - CORRECTED CRIME COUNT CALCULATION
# Matches Stage 6 spatial/temporal definitions
# ============================================================

import numpy as np
import time

print("Rebuilding Stage 8 crime counts...")

# ------------------------------------------------------------
# 1. Recreate outage geometries exactly as Stage 6
# ------------------------------------------------------------

outages_stage8 = gpd.read_parquet(OUTAGE_FILE)

# Same CRS sequence used in Stage 6:
# WGS84 -> EPSG:2263
outages_stage8 = outages_stage8.to_crs("EPSG:4326")
outages_stage8 = outages_stage8.to_crs("EPSG:2263")

# ------------------------------------------------------------
# 2. Make sure crime data is EPSG:2263
# ------------------------------------------------------------

crime_stage8 = gpd.read_parquet(CRIME_FILE)

crime_stage8["crime_datetime"] = pd.to_datetime(
    crime_stage8["crime_datetime"],
    errors="coerce"
)

crime_stage8 = crime_stage8.dropna(
    subset=["crime_datetime", "geometry"]
).copy()

crime_stage8 = crime_stage8.to_crs("EPSG:2263")

# ------------------------------------------------------------
# 3. Create crime spatial index using KDTree
# ------------------------------------------------------------

crime_x = crime_stage8.geometry.x.to_numpy()
crime_y = crime_stage8.geometry.y.to_numpy()

crime_coords = np.column_stack(
    [crime_x, crime_y]
)

crime_tree = cKDTree(crime_coords)

crime_times = crime_stage8["crime_datetime"].to_numpy()

print("Crime records:", len(crime_stage8))


# ------------------------------------------------------------
# 4. Prepare outage coordinates
# ------------------------------------------------------------

outage_xy = {}

for _, r in outages_stage8[
    ["unique_key", "geometry"]
].iterrows():

    if r["geometry"] is not None:
        outage_xy[int(r["unique_key"])] = (
            r["geometry"].x,
            r["geometry"].y
        )

print(
    "Unique outage locations:",
    len(outage_xy)
)


# ------------------------------------------------------------
# 5. Find crimes within 250m for each unique location
# ------------------------------------------------------------

location_keys = np.unique(
    np.concatenate([
        pairs["treatment_key"].astype(int).unique(),
        pairs["control_key"].astype(int).unique()
    ])
)

nearby_crimes_stage8 = {}

start = time.time()

for i, key in enumerate(location_keys):

    coord = outage_xy.get(int(key))

    if coord is None:
        nearby_crimes_stage8[int(key)] = np.array(
            [],
            dtype=int
        )
        continue

    nearby_crimes_stage8[int(key)] = np.asarray(
        crime_tree.query_ball_point(
            coord,
            r=250
        ),
        dtype=int
    )

    if (i + 1) % 5000 == 0:
        print(
            f"Processed {i + 1:,} / "
            f"{len(location_keys):,} locations"
        )

print(
    f"Spatial lookup completed in "
    f"{time.time() - start:.1f} seconds"
)


# ------------------------------------------------------------
# 6. Crime-count function
# ------------------------------------------------------------

def count_crimes_stage8(
    location_key,
    location_coord,
    start_time,
    end_time,
    start_inclusive=True,
    end_inclusive=False
):

    indices = nearby_crimes_stage8.get(
        int(location_key),
        np.array([], dtype=int)
    )

    if len(indices) == 0:
        return 0, 0

    times = crime_times[indices]

    start_np = np.datetime64(start_time)
    end_np = np.datetime64(end_time)

    if start_inclusive:
        start_mask = times >= start_np
    else:
        start_mask = times > start_np

    if end_inclusive:
        end_mask = times <= end_np
    else:
        end_mask = times < end_np

    time_mask = start_mask & end_mask

    if not time_mask.any():
        return 0, 0

    selected = indices[time_mask]

    dx = (
        crime_x[selected]
        - location_coord[0]
    )

    dy = (
        crime_y[selected]
        - location_coord[1]
    )

    distances = np.sqrt(
        dx ** 2 + dy ** 2
    )

    # Direct treatment zone: 0-100m
    direct = int(
        (distances <= 100).sum()
    )

    # Displacement zone: 100-250m
    displacement = int(
        (
            (distances > 100) &
            (distances <= 250)
        ).sum()
    )

    return direct, displacement


# ------------------------------------------------------------
# 7. Calculate PRE / DURING / POST for all pairs
# ------------------------------------------------------------

print("\nCalculating corrected causal crime counts...")

start = time.time()

corrected_results = []

for counter, (_, row) in enumerate(pairs.iterrows(), start=1):

    treatment_key = int(row["treatment_key"])
    control_key = int(row["control_key"])

    treatment_coord = outage_xy[treatment_key]
    control_coord = outage_xy[control_key]

    # --------------------------------------------------------
    # PRE
    # created_date - 14 days <= crime < created_date
    # --------------------------------------------------------

    treatment_pre_100, treatment_pre_250 = count_crimes_stage8(
        treatment_key,
        treatment_coord,
        row["pre_start"],
        row["pre_end"],
        start_inclusive=True,
        end_inclusive=False
    )

    control_pre_100, control_pre_250 = count_crimes_stage8(
        control_key,
        control_coord,
        row["pre_start"],
        row["pre_end"],
        start_inclusive=True,
        end_inclusive=False
    )

    # --------------------------------------------------------
    # DURING
    # created_date <= crime <= closed_date
    # Same definition as Stage 6
    # --------------------------------------------------------

    treatment_during_100, treatment_during_250 = count_crimes_stage8(
        treatment_key,
        treatment_coord,
        row["during_start"],
        row["during_end"],
        start_inclusive=True,
        end_inclusive=True
    )

    control_during_100, control_during_250 = count_crimes_stage8(
        control_key,
        control_coord,
        row["during_start"],
        row["during_end"],
        start_inclusive=True,
        end_inclusive=True
    )

    # --------------------------------------------------------
    # POST
    # closed_date < crime <= closed_date + 14 days
    # --------------------------------------------------------

    treatment_post_100, treatment_post_250 = count_crimes_stage8(
        treatment_key,
        treatment_coord,
        row["post_start"],
        row["post_end"],
        start_inclusive=False,
        end_inclusive=True
    )

    control_post_100, control_post_250 = count_crimes_stage8(
        control_key,
        control_coord,
        row["post_start"],
        row["post_end"],
        start_inclusive=False,
        end_inclusive=True
    )

    corrected_results.append({
        "treatment_pre_100m": treatment_pre_100,
        "treatment_pre_250m": treatment_pre_250,

        "treatment_during_100m": treatment_during_100,
        "treatment_during_250m": treatment_during_250,

        "treatment_post_100m": treatment_post_100,
        "treatment_post_250m": treatment_post_250,

        "control_pre_100m": control_pre_100,
        "control_pre_250m": control_pre_250,

        "control_during_100m": control_during_100,
        "control_during_250m": control_during_250,

        "control_post_100m": control_post_100,
        "control_post_250m": control_post_250,
    })

    if counter % 5000 == 0:
        print(
            f"Processed {counter:,} / "
            f"{len(pairs):,} pairs"
        )


# ------------------------------------------------------------
# 8. Replace old Stage 8 calculated counts
# ------------------------------------------------------------

corrected_counts = pd.DataFrame(
    corrected_results,
    index=pairs.index
)

count_columns = [
    "treatment_pre_100m",
    "treatment_pre_250m",
    "treatment_during_100m",
    "treatment_during_250m",
    "treatment_post_100m",
    "treatment_post_250m",
    "control_pre_100m",
    "control_pre_250m",
    "control_during_100m",
    "control_during_250m",
    "control_post_100m",
    "control_post_250m",
]

# Remove previous versions if they exist
pairs = pairs.drop(
    columns=[
        c for c in count_columns
        if c in pairs.columns
    ],
    errors="ignore"
)

# Add corrected counts
pairs = pd.concat(
    [
        pairs,
        corrected_counts[count_columns]
    ],
    axis=1
)

print(
    f"\nCorrected calculation completed in "
    f"{time.time() - start:.1f} seconds"
)

print("Final pair count:", len(pairs))

Rebuilding Stage 8 crime counts...
Crime records: 63195
Unique outage locations: 108030
Processed 5,000 / 46,124 locations
Processed 10,000 / 46,124 locations
Processed 15,000 / 46,124 locations
Processed 20,000 / 46,124 locations
Processed 25,000 / 46,124 locations
Processed 30,000 / 46,124 locations
Processed 35,000 / 46,124 locations
Processed 40,000 / 46,124 locations
Processed 45,000 / 46,124 locations
Spatial lookup completed in 0.2 seconds

Calculating corrected causal crime counts...
Processed 5,000 / 46,124 pairs
Processed 10,000 / 46,124 pairs
Processed 15,000 / 46,124 pairs
Processed 20,000 / 46,124 pairs
Processed 25,000 / 46,124 pairs
Processed 30,000 / 46,124 pairs
Processed 35,000 / 46,124 pairs
Processed 40,000 / 46,124 pairs
Processed 45,000 / 46,124 pairs

Corrected calculation completed in 3.1 seconds
Final pair count: 46124


In [53]:
# ============================================================
# VALIDATE CORRECTED STAGE 8 COUNTS
# ============================================================

print(
    "Treatment 100m mismatches:",
    (
        pairs["treatment_crime_100m"]
        != pairs["treatment_during_100m"]
    ).sum()
)

print(
    "Treatment 100-250m mismatches:",
    (
        pairs["treatment_crime_250m"]
        != pairs["treatment_during_250m"]
    ).sum()
)

Treatment 100m mismatches: 1
Treatment 100-250m mismatches: 1


In [54]:
# ============================================================
# STAGE 8 - SUMMARY OF CAUSAL CRIME COUNTS
# ============================================================

print("Pairs:", len(pairs))

print("\nTreatment:")
print(
    "PRE direct crimes:",
    pairs["treatment_pre_100m"].sum()
)
print(
    "DURING direct crimes:",
    pairs["treatment_during_100m"].sum()
)
print(
    "POST direct crimes:",
    pairs["treatment_post_100m"].sum()
)

print("\nTreatment displacement:")
print(
    "PRE 100-250m:",
    pairs["treatment_pre_250m"].sum()
)
print(
    "DURING 100-250m:",
    pairs["treatment_during_250m"].sum()
)
print(
    "POST 100-250m:",
    pairs["treatment_post_250m"].sum()
)

print("\nControl:")
print(
    "PRE direct crimes:",
    pairs["control_pre_100m"].sum()
)
print(
    "DURING direct crimes:",
    pairs["control_during_100m"].sum()
)
print(
    "POST direct crimes:",
    pairs["control_post_100m"].sum()
)

print("\nControl displacement:")
print(
    "PRE 100-250m:",
    pairs["control_pre_250m"].sum()
)
print(
    "DURING 100-250m:",
    pairs["control_during_250m"].sum()
)
print(
    "POST 100-250m:",
    pairs["control_post_250m"].sum()
)

Pairs: 46124

Treatment:
PRE direct crimes: 208
DURING direct crimes: 284
POST direct crimes: 225

Treatment displacement:
PRE 100-250m: 1107
DURING 100-250m: 1643
POST 100-250m: 1011

Control:
PRE direct crimes: 280
DURING direct crimes: 404
POST direct crimes: 236

Control displacement:
PRE 100-250m: 1078
DURING 100-250m: 1684
POST 100-250m: 1035


In [55]:
# ============================================================
# STAGE 8 - CONSTRUCT BALANCED CAUSAL PANEL
# ============================================================

print("Constructing balanced causal panel...")

# ------------------------------------------------------------
# 1. Create pair IDs
# ------------------------------------------------------------

pairs = pairs.reset_index(drop=True).copy()

pairs["pair_id"] = np.arange(
    1,
    len(pairs) + 1
)


# ------------------------------------------------------------
# 2. Build treatment observations
# ------------------------------------------------------------

treatment_panel = pd.DataFrame({
    "pair_id": pairs["pair_id"],

    "location_key": pairs["treatment_key"],

    "treatment": 1,

    "borough": pairs["treatment_borough"],

    "outage_duration_hours":
        pairs["treatment_duration_hours"],

    "h3_res9":
        pairs["h3_res9"],

    "h3_res10":
        pairs["h3_res10"],

    # PRE
    "crime_100m_pre":
        pairs["treatment_pre_100m"],

    "crime_250m_pre":
        pairs["treatment_pre_250m"],

    # DURING
    "crime_100m_during":
        pairs["treatment_during_100m"],

    "crime_250m_during":
        pairs["treatment_during_250m"],

    # POST
    "crime_100m_post":
        pairs["treatment_post_100m"],

    "crime_250m_post":
        pairs["treatment_post_250m"],
})


# ------------------------------------------------------------
# 3. Build control observations
# ------------------------------------------------------------

control_panel = pd.DataFrame({
    "pair_id": pairs["pair_id"],

    "location_key": pairs["control_key"],

    "treatment": 0,

    "borough": pairs["control_borough"],

    # Use the treatment outage duration because the
    # control is evaluated over the same treatment window.
    "outage_duration_hours":
        pairs["treatment_duration_hours"],

    "h3_res9":
        pairs["h3_res9_right"],

    "h3_res10":
        pairs["h3_res10_right"],

    # PRE
    "crime_100m_pre":
        pairs["control_pre_100m"],

    "crime_250m_pre":
        pairs["control_pre_250m"],

    # DURING
    "crime_100m_during":
        pairs["control_during_100m"],

    "crime_250m_during":
        pairs["control_during_250m"],

    # POST
    "crime_100m_post":
        pairs["control_post_100m"],

    "crime_250m_post":
        pairs["control_post_250m"],
})


# ------------------------------------------------------------
# 4. Combine treatment and control observations
# ------------------------------------------------------------

panel_wide = pd.concat(
    [
        treatment_panel,
        control_panel
    ],
    ignore_index=True
)

print(
    "Wide panel rows:",
    len(panel_wide)
)


# ------------------------------------------------------------
# 5. Convert to long format
# ------------------------------------------------------------

panel_rows = []

for _, row in panel_wide.iterrows():

    # --------------------------------------------------------
    # PRE
    # --------------------------------------------------------

    panel_rows.append({
        "pair_id": row["pair_id"],
        "location_key": row["location_key"],
        "treatment": row["treatment"],
        "borough": row["borough"],
        "outage_duration_hours":
            row["outage_duration_hours"],
        "h3_res9": row["h3_res9"],
        "h3_res10": row["h3_res10"],

        "period": "pre",
        "period_order": 0,

        "post": 0,

        "crime_100m":
            row["crime_100m_pre"],

        "crime_250m":
            row["crime_250m_pre"],
    })

    # --------------------------------------------------------
    # DURING
    # --------------------------------------------------------

    panel_rows.append({
        "pair_id": row["pair_id"],
        "location_key": row["location_key"],
        "treatment": row["treatment"],
        "borough": row["borough"],
        "outage_duration_hours":
            row["outage_duration_hours"],
        "h3_res9": row["h3_res9"],
        "h3_res10": row["h3_res10"],

        "period": "during",
        "period_order": 1,

        "post": 1,

        "crime_100m":
            row["crime_100m_during"],

        "crime_250m":
            row["crime_250m_during"],
    })

    # --------------------------------------------------------
    # POST
    # --------------------------------------------------------

    panel_rows.append({
        "pair_id": row["pair_id"],
        "location_key": row["location_key"],
        "treatment": row["treatment"],
        "borough": row["borough"],
        "outage_duration_hours":
            row["outage_duration_hours"],
        "h3_res9": row["h3_res9"],
        "h3_res10": row["h3_res10"],

        "period": "post",
        "period_order": 2,

        "post": 1,

        "crime_100m":
            row["crime_100m_post"],

        "crime_250m":
            row["crime_250m_post"],
    })


causal_panel = pd.DataFrame(panel_rows)


# ------------------------------------------------------------
# 6. Treatment × Post interaction
# ------------------------------------------------------------

causal_panel["treatment_x_post"] = (
    causal_panel["treatment"]
    *
    causal_panel["post"]
)


# ------------------------------------------------------------
# 7. Baseline crime intensity
# ------------------------------------------------------------

# Baseline intensity is the pre-outage direct crime count
# for the corresponding location/pair.

baseline = (
    causal_panel[
        causal_panel["period"] == "pre"
    ][
        [
            "pair_id",
            "crime_100m"
        ]
    ]
    .rename(
        columns={
            "crime_100m":
                "baseline_crime_intensity"
        }
    )
)

causal_panel = causal_panel.merge(
    baseline,
    on="pair_id",
    how="left"
)


# ------------------------------------------------------------
# 8. Add treatment outage dates
# ------------------------------------------------------------

pair_dates = pairs[
    [
        "pair_id",
        "created_date",
        "closed_date"
    ]
].copy()

causal_panel = causal_panel.merge(
    pair_dates,
    on="pair_id",
    how="left"
)


# ------------------------------------------------------------
# 9. Sort panel
# ------------------------------------------------------------

causal_panel = causal_panel.sort_values(
    [
        "pair_id",
        "treatment",
        "period_order"
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# 10. Final column order
# ------------------------------------------------------------

causal_panel = causal_panel[
    [
        "pair_id",
        "location_key",

        "created_date",
        "closed_date",

        "period",
        "period_order",

        "treatment",
        "post",
        "treatment_x_post",

        "crime_100m",
        "crime_250m",

        "baseline_crime_intensity",

        "borough",
        "outage_duration_hours",

        "h3_res9",
        "h3_res10",
    ]
]


# ------------------------------------------------------------
# 11. Save causal panel
# ------------------------------------------------------------

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

causal_panel.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("\n========================================")
print("       CAUSAL PANEL CREATED")
print("========================================")
print("Rows:", len(causal_panel))
print("Columns:", len(causal_panel.columns))
print("Pairs:", causal_panel["pair_id"].nunique())
print(
    "Unique locations:",
    causal_panel["location_key"].nunique()
)
print(
    "Periods:",
    causal_panel["period"].unique()
)
print(
    "Treatment observations:",
    (causal_panel["treatment"] == 1).sum()
)
print(
    "Control observations:",
    (causal_panel["treatment"] == 0).sum()
)
print(
    "Saved to:",
    OUTPUT_FILE
)
print("========================================")

Constructing balanced causal panel...
Wide panel rows: 92248

       CAUSAL PANEL CREATED
Rows: 553488
Columns: 16
Pairs: 46124
Unique locations: 46124
Periods: <ArrowStringArray>
['pre', 'during', 'post']
Length: 3, dtype: str
Treatment observations: 276744
Control observations: 276744
Saved to: ..\data\processed\causal_panel.parquet


In [56]:
# ============================================================
# FIX CAUSAL PANEL DUPLICATION
# ============================================================

print("Fixing causal panel structure...")

# ------------------------------------------------------------
# 1. Rebuild the panel from the original pair data
# ------------------------------------------------------------

treatment_panel = pd.DataFrame({
    "pair_id": pairs["pair_id"],
    "location_key": pairs["treatment_key"],
    "treatment": 1,
    "borough": pairs["treatment_borough"],
    "outage_duration_hours":
        pairs["treatment_duration_hours"],
    "h3_res9": pairs["h3_res9"],
    "h3_res10": pairs["h3_res10"],

    "crime_100m_pre":
        pairs["treatment_pre_100m"],
    "crime_250m_pre":
        pairs["treatment_pre_250m"],

    "crime_100m_during":
        pairs["treatment_during_100m"],
    "crime_250m_during":
        pairs["treatment_during_250m"],

    "crime_100m_post":
        pairs["treatment_post_100m"],
    "crime_250m_post":
        pairs["treatment_post_250m"],
})


control_panel = pd.DataFrame({
    "pair_id": pairs["pair_id"],
    "location_key": pairs["control_key"],
    "treatment": 0,
    "borough": pairs["control_borough"],

    # Controls use the treatment outage duration
    # because both locations are evaluated over
    # the same treatment calendar window.
    "outage_duration_hours":
        pairs["treatment_duration_hours"],

    "h3_res9": pairs["h3_res9_right"],
    "h3_res10": pairs["h3_res10_right"],

    "crime_100m_pre":
        pairs["control_pre_100m"],
    "crime_250m_pre":
        pairs["control_pre_250m"],

    "crime_100m_during":
        pairs["control_during_100m"],
    "crime_250m_during":
        pairs["control_during_250m"],

    "crime_100m_post":
        pairs["control_post_100m"],
    "crime_250m_post":
        pairs["control_post_250m"],
})


# ------------------------------------------------------------
# 2. Combine treatment and control
# ------------------------------------------------------------

panel_wide = pd.concat(
    [
        treatment_panel,
        control_panel
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 3. Convert each location to 3 periods
# ------------------------------------------------------------

panel_rows = []

for _, row in panel_wide.iterrows():

    for period, period_order, post, crime100, crime250 in [
        (
            "pre",
            0,
            0,
            row["crime_100m_pre"],
            row["crime_250m_pre"]
        ),
        (
            "during",
            1,
            1,
            row["crime_100m_during"],
            row["crime_250m_during"]
        ),
        (
            "post",
            2,
            1,
            row["crime_100m_post"],
            row["crime_250m_post"]
        )
    ]:

        panel_rows.append({
            "pair_id": row["pair_id"],
            "location_key": row["location_key"],

            "treatment": row["treatment"],

            "borough": row["borough"],

            "outage_duration_hours":
                row["outage_duration_hours"],

            "h3_res9": row["h3_res9"],
            "h3_res10": row["h3_res10"],

            "period": period,
            "period_order": period_order,

            "post": post,

            "crime_100m": crime100,
            "crime_250m": crime250,
        })


causal_panel = pd.DataFrame(panel_rows)


# ------------------------------------------------------------
# 4. Treatment × Post interaction
# ------------------------------------------------------------

causal_panel["treatment_x_post"] = (
    causal_panel["treatment"]
    *
    causal_panel["post"]
)


# ------------------------------------------------------------
# 5. Baseline crime intensity
#
# IMPORTANT:
# Match baseline by BOTH pair_id and treatment.
# This prevents treatment and control PRE rows
# from being mixed together.
# ------------------------------------------------------------

baseline = (
    causal_panel[
        causal_panel["period"] == "pre"
    ][
        [
            "pair_id",
            "treatment",
            "crime_100m"
        ]
    ]
    .rename(
        columns={
            "crime_100m":
                "baseline_crime_intensity"
        }
    )
)

causal_panel = causal_panel.merge(
    baseline,
    on=[
        "pair_id",
        "treatment"
    ],
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 6. Add outage dates
# ------------------------------------------------------------

pair_dates = pairs[
    [
        "pair_id",
        "created_date",
        "closed_date"
    ]
].copy()

causal_panel = causal_panel.merge(
    pair_dates,
    on="pair_id",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 7. Sort
# ------------------------------------------------------------

causal_panel = causal_panel.sort_values(
    [
        "pair_id",
        "treatment",
        "period_order"
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# 8. Final column order
# ------------------------------------------------------------

causal_panel = causal_panel[
    [
        "pair_id",
        "location_key",

        "created_date",
        "closed_date",

        "period",
        "period_order",

        "treatment",
        "post",
        "treatment_x_post",

        "crime_100m",
        "crime_250m",

        "baseline_crime_intensity",

        "borough",
        "outage_duration_hours",

        "h3_res9",
        "h3_res10",
    ]
]


# ------------------------------------------------------------
# 9. Save corrected panel
# ------------------------------------------------------------

causal_panel.to_parquet(
    OUTPUT_FILE,
    index=False
)


# ------------------------------------------------------------
# 10. Structural validation
# ------------------------------------------------------------

print("\n========================================")
print("       CORRECTED CAUSAL PANEL")
print("========================================")

print("Rows:", len(causal_panel))
print("Columns:", len(causal_panel.columns))

print(
    "Unique pairs:",
    causal_panel["pair_id"].nunique()
)

print(
    "Unique locations:",
    causal_panel["location_key"].nunique()
)

print(
    "Treatment rows:",
    (causal_panel["treatment"] == 1).sum()
)

print(
    "Control rows:",
    (causal_panel["treatment"] == 0).sum()
)

print(
    "PRE rows:",
    (causal_panel["period"] == "pre").sum()
)

print(
    "DURING rows:",
    (causal_panel["period"] == "during").sum()
)

print(
    "POST rows:",
    (causal_panel["period"] == "post").sum()
)

print(
    "Missing baseline:",
    causal_panel["baseline_crime_intensity"].isna().sum()
)

print(
    "Saved to:",
    OUTPUT_FILE
)

print("========================================")

Fixing causal panel structure...

       CORRECTED CAUSAL PANEL
Rows: 276744
Columns: 16
Unique pairs: 46124
Unique locations: 46124
Treatment rows: 138372
Control rows: 138372
PRE rows: 92248
DURING rows: 92248
POST rows: 92248
Missing baseline: 0
Saved to: ..\data\processed\causal_panel.parquet


In [57]:
# ============================================================
# FINAL STAGE 8 VALIDATION
# ============================================================

print("Treatment values:")
print(causal_panel["treatment"].value_counts().sort_index())

print("\nPost values:")
print(causal_panel["post"].value_counts().sort_index())

print("\nTreatment × Post:")
print(causal_panel["treatment_x_post"].value_counts().sort_index())

print("\nPeriod × Treatment:")
print(
    causal_panel
    .groupby(["period", "treatment"])
    .size()
)

print("\nMissing values:")
print(
    causal_panel.isna().sum()
)

print("\nFirst 10 rows:")
display(causal_panel.head(10))

Treatment values:
treatment
0    138372
1    138372
Name: count, dtype: int64

Post values:
post
0     92248
1    184496
Name: count, dtype: int64

Treatment × Post:
treatment_x_post
0    184496
1     92248
Name: count, dtype: int64

Period × Treatment:
period  treatment
during  0            46124
        1            46124
post    0            46124
        1            46124
pre     0            46124
        1            46124
dtype: int64

Missing values:
pair_id                     0
location_key                0
created_date                0
closed_date                 0
period                      0
period_order                0
treatment                   0
post                        0
treatment_x_post            0
crime_100m                  0
crime_250m                  0
baseline_crime_intensity    0
borough                     0
outage_duration_hours       0
h3_res9                     0
h3_res10                    0
dtype: int64

First 10 rows:


,pair_id,location_key,created_date,closed_date,period,period_order,treatment,post,treatment_x_post,crime_100m,crime_250m,baseline_crime_intensity,borough,outage_duration_hours,h3_res9,h3_res10
0,1,48296388,2020-01-01 11:04:00,2020-01-04 00:05:00,pre,0,0,0,0,0,0,0,MANHATTAN,61.016667,892a1072dc3ffff,8a2a1072dc1ffff
1,1,48296388,2020-01-01 11:04:00,2020-01-04 00:05:00,during,1,0,1,0,0,0,0,MANHATTAN,61.016667,892a1072dc3ffff,8a2a1072dc1ffff
2,1,48296388,2020-01-01 11:04:00,2020-01-04 00:05:00,post,2,0,1,0,0,0,0,MANHATTAN,61.016667,892a1072dc3ffff,8a2a1072dc1ffff
3,1,45284934,2020-01-01 11:04:00,2020-01-04 00:05:00,pre,0,1,0,0,0,0,0,MANHATTAN,61.016667,892a1072dcfffff,8a2a1072dce7fff
4,1,45284934,2020-01-01 11:04:00,2020-01-04 00:05:00,during,1,1,1,1,0,0,0,MANHATTAN,61.016667,892a1072dcfffff,8a2a1072dce7fff
5,1,45284934,2020-01-01 11:04:00,2020-01-04 00:05:00,post,2,1,1,1,0,0,0,MANHATTAN,61.016667,892a1072dcfffff,8a2a1072dce7fff
6,2,64135128,2020-01-01 06:55:00,2020-01-03 10:54:00,pre,0,0,0,0,0,0,0,MANHATTAN,51.983333,892a1072137ffff,8a2a1072134ffff
7,2,64135128,2020-01-01 06:55:00,2020-01-03 10:54:00,during,1,0,1,0,0,0,0,MANHATTAN,51.983333,892a1072137ffff,8a2a1072134ffff
8,2,64135128,2020-01-01 06:55:00,2020-01-03 10:54:00,post,2,0,1,0,0,0,0,MANHATTAN,51.983333,892a1072137ffff,8a2a1072134ffff
9,2,45284935,2020-01-01 06:55:00,2020-01-03 10:54:00,pre,0,1,0,0,0,0,0,MANHATTAN,51.983333,892a1072137ffff,8a2a1072134ffff
